In [1]:
import os
import json
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from sklearn.preprocessing import (MaxAbsScaler, MinMaxScaler, 
                                StandardScaler, RobustScaler)

def load_parquet_in_chunks(file_path, chunk_size=100000):
    """
    Carga un archivo parquet en chunks para manejar mejor la memoria.
    """
    parquet_file = pq.ParquetFile(file_path)
    
    # Leer el primer chunk para obtener las columnas
    first_chunk = next(parquet_file.iter_batches(batch_size=chunk_size))
    df = first_chunk.to_pandas()
    
    # Leer los chunks restantes y concatenarlos
    remaining_chunks = []
    for batch in parquet_file.iter_batches(batch_size=chunk_size):
        chunk_df = batch.to_pandas()
        remaining_chunks.append(chunk_df)
    
    if remaining_chunks:
        df = pd.concat([df] + remaining_chunks, ignore_index=True)
    
    return df

def is_boolean_column(series):
    """Detecta si una columna es booleana (solo contiene 0 y 1)."""
    unique_values = series.dropna().unique()
    return set(unique_values).issubset({0, 1})

def get_columns_to_normalize(df):
    """Identifica las columnas a normalizar: todas excepto booleanas y target."""
    columns_to_normalize = []
    
    for col in df.columns:
        if col == 'nivel_triage':  # Excluir target
            continue
        
        # Verificar si la columna es numérica
        if np.issubdtype(df[col].dtype, np.number):
            # Verificar si no es booleana
            if not is_boolean_column(df[col]):
                columns_to_normalize.append(col)
    
    return columns_to_normalize

def print_dataset_info(df, stage):
    """Imprime información sobre las columnas del dataset."""
    print(f"\n📊 {stage}")
    print("-" * 50)
    print(f"Número total de columnas: {len(df.columns)}")
    print("\nTipos de datos:")
    for dtype, count in df.dtypes.value_counts().items():
        print(f"  - {dtype}: {count} columnas")
    
    # Conteo de columnas booleanas
    boolean_cols = [col for col in df.columns if is_boolean_column(df[col])]
    print(f"\nColumnas booleanas detectadas: {len(boolean_cols)}")
    
    return {
        'total_columns': len(df.columns),
        'dtype_counts': df.dtypes.value_counts().to_dict(),
        'boolean_cols': len(boolean_cols)
    }

def validate_columns(df_original, df_normalized, method):
    """Valida que las columnas sean consistentes después de la normalización."""
    original_cols = set(df_original.columns)
    normalized_cols = set(df_normalized.columns)
    
    print(f"\n🔍 Validación de columnas para {method}:")
    print("-" * 50)
    
    if original_cols == normalized_cols:
        print("✅ Las columnas se mantienen idénticas después de la normalización")
    else:
        print("❌ Se detectaron diferencias en las columnas:")
        added = normalized_cols - original_cols
        removed = original_cols - normalized_cols
        if added:
            print(f"  Columnas agregadas: {added}")
        if removed:
            print(f"  Columnas eliminadas: {removed}")
    
    return original_cols == normalized_cols

def apply_normalization(df, columns_to_normalize, method):
    """Aplica el método de normalización especificado a las columnas seleccionadas."""
    scalers = {
        "MaxAbs": MaxAbsScaler(),
        "MinMax": MinMaxScaler(),
        "Standard": StandardScaler(),
        "Robust": RobustScaler(),
        "None": None
    }
    
    if scalers[method]:
        # Crear una copia para evitar problemas de memoria
        subset = df[columns_to_normalize].copy().astype('float32')
        
        # Aplicar el scaler en chunks para evitar problemas de memoria
        chunk_size = 50  # Procesar 50 columnas a la vez
        for i in range(0, len(columns_to_normalize), chunk_size):
            chunk_cols = columns_to_normalize[i:i+chunk_size]
            df[chunk_cols] = scalers[method].fit_transform(df[chunk_cols])
    
    return df

def save_dataframe(df, output_file, chunk_size=100000):
    """
    Guarda un DataFrame en formato parquet optimizando la memoria.
    """
    try:
        # Guardar en chunks para evitar problemas de memoria
        for i in range(0, len(df), chunk_size):
            chunk_df = df.iloc[i:i+chunk_size]
            mode = 'w' if i == 0 else 'a'
            chunk_df.to_parquet(output_file, index=False, engine='pyarrow', compression='gzip', mode=mode)
        print(f"✅ Dataset guardado exitosamente en: {output_file}")
    except Exception as e:
        print(f"Error al guardar el archivo: {str(e)}")
        
        # Si falla el método anterior, intentar con un enfoque alternativo
        try:
            import pyarrow as pa
            print("Intentando método alternativo de guardado...")
            # Dividir en chunks más pequeños
            smaller_chunk_size = chunk_size // 2
            for i in range(0, len(df), smaller_chunk_size):
                chunk_df = df.iloc[i:i+smaller_chunk_size]
                chunk_table = pa.Table.from_pandas(chunk_df)
                
                if i == 0:
                    pq.write_table(chunk_table, output_file)
                else:
                    pq.write_table(chunk_table, output_file, append=True)
            print(f"✅ Dataset guardado exitosamente con método alternativo.")
        except Exception as e2:
            print(f"Error también en método alternativo: {str(e2)}")
            raise

def main():
    try:
        # En notebooks, no tenemos __file__, así que usamos getcwd() como alternativa
        script_dir = os.getcwd()
        base_path = os.path.dirname(script_dir)
        
        # Imprimir las rutas para verificar
        print(f"Script directory: {script_dir}")
        print(f"Base path: {base_path}")
        
        # Intentar encontrar el archivo de configuración
        config_paths = [
            os.path.join(base_path, "config.json"),  # Opción 1: En el directorio padre
            os.path.join(script_dir, "config.json"), # Opción 2: En el directorio actual
            os.path.join(script_dir, "..", "config.json")  # Opción 3: Una carpeta arriba
        ]
        
        config = None
        for path in config_paths:
            if os.path.exists(path):
                with open(path, "r") as f:
                    config = json.load(f)
                print(f"Configuración cargada correctamente de: {path}")
                config_path = path
                break
        
        if config is None:
            # Ruta absoluta al directorio del proyecto
            project_dir = os.path.join(os.path.dirname(os.path.dirname(script_dir)))
            config_path = os.path.join(project_dir, "tesis_austral_2", "config.json")
            
            if os.path.exists(config_path):
                with open(config_path, "r") as f:
                    config = json.load(f)
                print(f"Configuración cargada correctamente de: {config_path}")
            else:
                raise FileNotFoundError(f"No se pudo encontrar el archivo de configuración en ninguna de las rutas probadas")
        
        # Configuración ajustada a la nueva estructura
        # Ajustamos las rutas para apuntar a las ubicaciones correctas
        base_github_path = os.path.abspath(os.path.join(os.path.expanduser("~"), "Documents", "GitHub", "TesisAustral"))
        
        # Rutas corregidas
        input_path = os.path.join(base_github_path, "Archivos_tesis", "intermediate", "featured")
        output_path = os.path.join(base_github_path, "Archivos_tesis", "intermediate", "normalized")
        
        print(f"\nRutas configuradas:")
        print(f"- Input path: {input_path}")
        print(f"- Output path: {output_path}")
        
        os.makedirs(output_path, exist_ok=True)
        
        # Cargar dataset con feature engineering usando chunks
        input_file = os.path.join(input_path, "df_feateng.parquet")
        print(f"\n📂 Cargando dataset desde: {input_file}")
        
        # Verificar si el archivo existe
        if not os.path.exists(input_file):
            # Buscar en ubicaciones alternativas
            alternative_paths = [
                os.path.join(input_path, "df_feateng_train.parquet"),  # Buscar train en lugar de completo
                os.path.join(base_github_path, "tesis_austral_2", "data", "df_feateng.parquet"),  # Otra posible ubicación
                os.path.join(base_github_path, "data", "df_feateng.parquet")  # Otra posible ubicación
            ]
            
            found = False
            for alt_path in alternative_paths:
                if os.path.exists(alt_path):
                    input_file = alt_path
                    print(f"Archivo encontrado en ubicación alternativa: {input_file}")
                    found = True
                    break
            
            if not found:
                raise FileNotFoundError(f"No se encontró el archivo {input_file} ni en ubicaciones alternativas")
        
        # Cargar el dataset en chunks para evitar problemas de memoria
        df = load_parquet_in_chunks(input_file)
        
        # Información del dataset original
        original_info = print_dataset_info(df, "Dataset Original")
        
        # Identificar columnas a normalizar
        columns_to_normalize = get_columns_to_normalize(df)
        
        print(f"\n🎯 Columnas identificadas para normalización:")
        print(f"Total columnas a normalizar: {len(columns_to_normalize)}")
        print(f"Representa el {(len(columns_to_normalize)/len(df.columns))*100:.2f}% del total de columnas")
        print("\nDesglose de columnas excluidas:")
        print(f"- Target (nivel_triage)")
        print(f"- Columnas booleanas: {len([col for col in df.columns if is_boolean_column(df[col])])}")
        
        # Diccionario para almacenar resultados
        results = {}
        
        # Aplicar todas las normalizaciones (con manejo de memoria mejorado)
        for method in ["MaxAbs", "MinMax", "Standard", "Robust", "None"]:
            print(f"\n🔄 Aplicando normalización: {method}")
            print("-" * 50)
            
            # Crear copia profunda para evitar modificar el original
            df_norm = df.copy()
            
            try:
                # Aplicar normalización
                df_norm = apply_normalization(df_norm, columns_to_normalize, method)
                
                # Validar y guardar información
                norm_info = print_dataset_info(df_norm, f"Dataset Normalizado ({method})")
                columns_valid = validate_columns(df, df_norm, method)
                
                # Crear directorio para el método específico
                method_dir = os.path.join(output_path, method)
                os.makedirs(method_dir, exist_ok=True)
                
                # Guardar dataset normalizado
                output_file = os.path.join(method_dir, f"df_feateng_{method}.parquet")
                save_dataframe(df_norm, output_file)
                
                results[method] = {
                    'output_file': output_file,
                    'columns_valid': columns_valid,
                    'info': norm_info
                }
                
                # Liberar memoria
                del df_norm
                import gc
                gc.collect()
                
            except MemoryError:
                print(f"⚠️ Error de memoria al procesar {method}. Intentando con enfoque de división...")
                
                # Enfoque alternativo si hay problemas de memoria
                # Dividir en subconjuntos más pequeños
                subset_size = len(df) // 4  # Dividir en 4 partes
                
                output_file = os.path.join(output_path, f"df_feateng_{method}.parquet")
                
                for i in range(0, len(df), subset_size):
                    end_idx = min(i + subset_size, len(df))
                    print(f"Procesando subset {i//subset_size + 1}/4 ({i}:{end_idx})")
                    
                    # Procesar subconjunto
                    df_subset = df.iloc[i:end_idx].copy()
                    df_subset = apply_normalization(df_subset, columns_to_normalize, method)
                    
                    # Guardar subconjunto
                    mode = 'w' if i == 0 else 'a'
                    df_subset.to_parquet(output_file, index=False, engine='pyarrow', mode=mode)
                    
                    # Liberar memoria
                    del df_subset
                    gc.collect()
                
                print(f"✅ Dataset dividido guardado en: {output_file}")
                
                # Registrar resultado
                results[method] = {
                    'output_file': output_file,
                    'columns_valid': True,  # Asumimos que es válido
                    'info': {'total_columns': len(df.columns)}  # Info mínima
                }
            
            except Exception as e:
                print(f"\n❌ Error durante la normalización {method}: {str(e)}")
                print("Continuando con el siguiente método...")
        
        # Resumen final
        print("\n📑 Resumen Final")
        print("-" * 50)
        print(f"Dataset original: {len(df.columns)} columnas")
        print(f"Columnas normalizadas: {len(columns_to_normalize)}")
        for method, result in results.items():
            print(f"\n{method}:")
            print(f"  - Columnas totales: {result['info'].get('total_columns', 'N/A')}")
            print(f"  - Validación: {'✅' if result.get('columns_valid', False) else '❌'}")
            print(f"  - Archivo: {os.path.basename(result['output_file'])}")
            
    except Exception as e:
        print(f"\n❌ Error durante la ejecución: {str(e)}")
        import traceback
        traceback.print_exc()
        raise

if __name__ == "__main__":
    main()

Script directory: c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2
Base path: c:\Users\Gonzalo\Documents\GitHub\TesisAustral
Configuración cargada correctamente de: c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\config.json

Rutas configuradas:
- Input path: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featured
- Output path: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\normalized

📂 Cargando dataset desde: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featured\df_feateng.parquet

📊 Dataset Original
--------------------------------------------------
Número total de columnas: 713

Tipos de datos:
  - float64: 712 columnas
  - int32: 1 columnas

Columnas booleanas detectadas: 125

🎯 Columnas identificadas para normalización:
Total columnas a normalizar: 587
Representa el 82.33% del total de columnas

Desglose de columnas excluidas:
- Target (nivel_triage)
- Columnas bool